In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

In [ ]:
# Load and preprocess the data for each patient
def load_and_preprocess_data(patient_directory, response_label):
    adata = sc.read_10x_mtx(patient_directory, var_names='gene_symbols', cache=True)
    adata.obs['patient'] = patient_directory
    adata.obs['response'] = response_label
    adata.obs = adata.obs.reset_index()
    adata.obs['index'] = adata.obs['index'] + '_' + adata.obs['response'].astype(str)
    adata.obs = adata.obs.set_index('index')
    adata.var_names_make_unique()
    return adata

adatap4 = load_and_preprocess_data('MD01-004_tumor_1', 'Non-MPR')
adatap5 = load_and_preprocess_data('MD01-005_tumor_2', 'MPR')
adatap10 = load_and_preprocess_data('MD01-010_tumor_1', 'MPR')
adatap19 = load_and_preprocess_data('MD01-019_tumor_1', 'Non-MPR')
adatap24 = load_and_preprocess_data('MD01-024_tumor_1', 'Non-MPR')
adatap3 = load_and_preprocess_data('MD043-003_tumor_1', 'MPR')
adatap6 = load_and_preprocess_data('MD043-006_tumor_1', 'Non-MPR')
adatap8 = load_and_preprocess_data('MD043-008_tumor_1', 'MPR')
adatap11 = load_and_preprocess_data('MD043-011_tumor_2', 'Non-MPR')
adatap7 = load_and_preprocess_data('NY016-007_tumor_1', 'Non-MPR')
adatap14 = load_and_preprocess_data('NY016-014_tumor_1', 'Non-MPR')
adatap15 = load_and_preprocess_data('NY016-015_tumor_1', 'Non-MPR')
adatap21 = load_and_preprocess_data('NY016-021_tumor_1', 'Non-MPR')
adatap22 = load_and_preprocess_data('NY016-022_tumor_1', 'MPR')
adatap25 = load_and_preprocess_data('NY016-025_tumor_1', 'MPR')

# Concatenate data from all patients
patients_data = sc.concat([adatap4, adatap5, adatap10, adatap19, adatap24, adatap3, adatap6, adatap8, adatap11, adatap7, adatap14, adatap15, adatap21, adatap22, adatap25])

In [ ]:
# Aggregate the data at the patient level for each patient
data_p4 = adatap4.X.sum(axis=0)
data_p5 = adatap5.X.sum(axis=0)
data_p10 = adatap10.X.sum(axis=0)
data_p19 = adatap19.X.sum(axis=0)
data_p24 = adatap24.X.sum(axis=0)
data_p3 = adatap3.X.sum(axis=0)
data_p6 = adatap6.X.sum(axis=0)
data_p8 = adatap8.X.sum(axis=0)
data_p11 = adatap11.X.sum(axis=0)
data_p7 = adatap7.X.sum(axis=0)
data_p14 = adatap14.X.sum(axis=0)
data_p15 = adatap15.X.sum(axis=0)
data_p21 = adatap21.X.sum(axis=0)
data_p22 = adatap22.X.sum(axis=0)
data_p25 = adatap25.X.sum(axis=0)

# Create labels for patients based on their responses
patient_labels = pd.Series(['Non-MPR','MPR','MPR','Non-MPR','Non-MPR','MPR','Non-MPR','MPR','Non-MPR','Non-MPR','Non-MPR','Non-MPR','Non-MPR','MPR','MPR'], index=['P4', 'P5', 'P10', 'P19', 'P24', 'P3', 'P6', 'P8', 'P11', 'P7', 'P14', 'P15', 'P21', 'P22', 'P25'])

# Combine the aggregated data into a DataFrame
aggregated_data = pd.DataFrame({
    'P4': data_p4.A1,
    'P5': data_p5.A1,
    'P10': data_p10.A1,
    'P19': data_p19.A1,
    'P24': data_p24.A1,
    'P3': data_p3.A1,
    'P6': data_p6.A1,
    'P8': data_p8.A1,
    'P11': data_p11.A1,
    'P7': data_p7.A1,
    'P14': data_p14.A1,
    'P15': data_p15.A1,
    'P21': data_p21.A1,
    'P22': data_p22.A1,
    'P25': data_p25.A1
})

# Transpose the DataFrame for the correct shape
aggregated_data = aggregated_data.T

# Split the data and labels into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(aggregated_data, patient_labels, test_size=0.2, random_state=42, stratify=patient_labels)

# Standardize features if necessary
scaler = StandardScaler(with_mean=False)
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Create and train a Random Forest classifier (you can choose a different classifier)
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Make predictions on the test set
y_pred = clf.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print("Accuracy:", accuracy)
print("Classification Report:\n", report)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Gradient Boosting Classifier
gb = GradientBoostingClassifier(random_state=42)
gb_param_grid = {
    'n_estimators': [1,2,3,4,5,6,7,8,9,10,15,20,25,50,75,100,125,150],
    'learning_rate': [0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1],
    'max_depth': [None, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100],
    'min_samples_split': [2, 3, 4]
}

gb_grid_search = GridSearchCV(estimator=gb, param_grid=gb_param_grid, cv=4, n_jobs=-1, scoring='accuracy')
gb_grid_search.fit(X_train_scaled, y_train)
gb_best = gb_grid_search.best_estimator_

# Random Forest Classifier
rf = RandomForestClassifier(random_state=42)
rf_param_grid = {
    'n_estimators': [1,2,3,4,5,6,7,8,9,10,15,20,25,50,75,100,125,150],
    'max_features': ['auto','sqrt','log2'],
    'max_depth': [None, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100],
    'min_samples_split': [2, 3, 4],
    'min_samples_leaf': [1,2,3,4,5,10]
}

rf_grid_search = GridSearchCV(estimator=rf, param_grid=rf_param_grid, cv=4, n_jobs=-1, scoring='accuracy')
rf_grid_search.fit(X_train, y_train)
rf_best = rf_grid_search.best_estimator_

# Cross-validation scores
gb_cv_scores = cross_val_score(gb_best, X_train_scaled, y_train, cv=4, scoring='accuracy')
rf_cv_scores = cross_val_score(rf_best, X_train, y_train, cv=4, scoring='accuracy')

# Evaluate the models on the test set
gb_test_pred = gb_best.predict(X_test_scaled)
rf_test_pred = rf_best.predict(X_test)

gb_test_acc = accuracy_score(y_test, gb_test_pred)
rf_test_acc = accuracy_score(y_test, rf_test_pred)

# Print results
print("Gradient Boosting Best Parameters:", gb_grid_search.best_params_)
print("Gradient Boosting Cross-Validation Accuracy:", gb_cv_scores.mean())
print("Gradient Boosting Test Accuracy:", gb_test_acc)

print("Random Forest Best Parameters:", rf_grid_search.best_params_)
print("Random Forest Cross-Validation Accuracy:", rf_cv_scores.mean())
print("Random Forest Test Accuracy:", rf_test_acc)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
# Cross-validation scores
gb_cv_scores = cross_val_score(gb_best, X_train_scaled, y_train, cv=4, scoring='accuracy')
rf_cv_scores = cross_val_score(rf_best, X_train, y_train, cv=4, scoring='accuracy')

# Evaluate the models on the test set
gb_test_pred = gb_best.predict(X_test_scaled)
rf_test_pred = rf_best.predict(X_test)

gb_test_acc = accuracy_score(y_test, gb_test_pred)
rf_test_acc = accuracy_score(y_test, rf_test_pred)

# Classification report
gb_class_report = classification_report(y_test, gb_test_pred, output_dict=True)
rf_class_report = classification_report(y_test, rf_test_pred, output_dict=True)

# Feature importance
gb_importances = gb_best.feature_importances_
rf_importances = rf_best.feature_importances_

# Create a DataFrame for feature importances
features = aggregated_data.columns
gb_importance_df = pd.DataFrame({'Feature': features, 'Importance': gb_importances}).sort_values(by='Importance', ascending=False)
rf_importance_df = pd.DataFrame({'Feature': features, 'Importance': rf_importances}).sort_values(by='Importance', ascending=False)

# Plot feature importances
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
sns.barplot(x='Importance', y='Feature', data=gb_importance_df)
plt.title('Gradient Boosting Feature Importances')

plt.subplot(1, 2, 2)
sns.barplot(x='Importance', y='Feature', data=rf_importance_df)
plt.title('Random Forest Feature Importances')

plt.tight_layout()
plt.show()

# Confusion matrix
gb_cm = confusion_matrix(y_test, gb_test_pred)
rf_cm = confusion_matrix(y_test, rf_test_pred)

# Plot confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(gb_cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Gradient Boosting Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(rf_cm, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title('Random Forest Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.show()

# Print results
print("Gradient Boosting Best Parameters:", gb_grid_search.best_params_)
print("Gradient Boosting Cross-Validation Accuracy:", gb_cv_scores.mean())
print("Gradient Boosting Test Accuracy:", gb_test_acc)
print("\nGradient Boosting Classification Report:\n", classification_report(y_test, gb_test_pred))

print("Random Forest Best Parameters:", rf_grid_search.best_params_)
print("Random Forest Cross-Validation Accuracy:", rf_cv_scores.mean())
print("Random Forest Test Accuracy:", rf_test_acc)
print("\nRandom Forest Classification Report:\n", classification_report(y_test, rf_test_pred))

In [ ]:
new_patient_data = load_and_preprocess_data('datahuman\P12', 'NA')
new_patient_data

In [ ]:
# Aggregate the data at the patient level for the new patient
new_patient_aggregated_data = new_patient_data.X.sum(axis=0).A1

# Standardize the features using the same scaler as before (assuming 'scaler' is the scaler used for training)
new_patient_aggregated_data_scaled = scaler.transform([new_patient_aggregated_data])

# Use the trained classifiers to make predictions for the new patient
prediction_rf_best = rf_best.predict(new_patient_aggregated_data_scaled)
prediction_gb_best = gb_best.predict(new_patient_aggregated_data_scaled)

# The 'prediction' variable now contains the predicted response for the new patient (either 'pCR' or 'non-pCR')
print("Predicted Response for New Patient:", prediction_rf_best[0])
print("Predicted Response for New Patient:", prediction_gb_best[0])